In [1]:
import pandas as pd

# 1. Load the basic order data
orders = pd.read_csv("olist_orders_dataset.csv")

# 2. THE NEW INGREDIENT! Load the customer reviews
reviews = pd.read_csv("olist_order_reviews_dataset.csv")

# Let's see how many reviews we have and what they look like
print("Number of reviews:", reviews.shape[0])
print("\n--- REVIEWS X-RAY (INFO) ---")
reviews.info()

Number of reviews: 99224

--- REVIEWS X-RAY (INFO) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [2]:
# 1. SIMPLIFY: Create the target column (Target)
# We use a quick function (lambda) to say: "If the score is greater than 2, put a 1, otherwise, put a 0"
reviews['happy_customer'] = reviews['review_score'].apply(lambda x: 1 if x > 2 else 0)

# 2. CONTEXT: Merge reviews with orders
ml_data = pd.merge(orders, reviews, on='order_id', how='inner')

# 3. FEATURE ENGINEERING: Create the "Shipping Days" column
# First, ensure Python understands these are dates
ml_data['order_purchase_timestamp'] = pd.to_datetime(ml_data['order_purchase_timestamp'])
ml_data['order_delivered_customer_date'] = pd.to_datetime(ml_data['order_delivered_customer_date'])

# Subtract purchase date from delivery date to get the days
ml_data['shipping_days'] = (ml_data['order_delivered_customer_date'] - ml_data['order_purchase_timestamp']).dt.days

# Clean up orders that don't have shipping days yet (nulls)
ml_data = ml_data.dropna(subset=['shipping_days'])

print("New clues created for the AI!")
print(ml_data[['review_score', 'happy_customer', 'shipping_days']].head(5))


New clues created for the AI!
   review_score  happy_customer  shipping_days
0             4               1            8.0
1             4               1           13.0
2             5               1            9.0
3             5               1           13.0
4             5               1            2.0


In [3]:
!pip install scikit-learn


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. DEFINE CLUES AND ANSWERS
X = ml_data[['shipping_days']]  # The CLUES (Features) the AI can look at
y = ml_data['happy_customer']   # The ANSWER (Target) the AI has to guess

# 2. SPLIT FOR THE EXAM (80% study, 20% exam)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"The AI will study with {X_train.shape[0]} historical orders.")
print(f"We will test it with {X_test.shape[0]} hidden orders.")

# 3. CREATE AND TRAIN THE MODEL (Here's where the magic happens)
print("\nTraining the Artificial Intelligence... (this may take a few seconds)")
model = RandomForestClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train) # The .fit() command is where the AI learns the mathematical patterns

print("Training successfully completed! 🚀")

The AI will study with 77087 historical orders.
We will test it with 19272 hidden orders.

Training the Artificial Intelligence... (this may take a few seconds)
Training successfully completed! 🚀


In [6]:
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. TAKE THE EXAM (Predictions)
# We give the AI the hidden clues (X_test) and ask it to guess the answer
predictions = model.predict(X_test)

# 2. GRADE THE EXAM (Accuracy)
# We compare what the AI guessed with the actual answers we had saved (y_test)
exam_score = accuracy_score(y_test, predictions)

print(f"Exam graded! The AI guessed correctly {exam_score * 100:.2f}% of the time.")

# 3. SEE WHERE IT MADE MISTAKES (The famous Confusion Matrix)
matrix = confusion_matrix(y_test, predictions)
print("\nConfusion Matrix (The detail of hits and misses):")
print(matrix)

Exam graded! The AI guessed correctly 88.36% of the time.

Confusion Matrix (The detail of hits and misses):
[[  514  1987]
 [  256 16515]]
